In [9]:
import warnings
warnings.filterwarnings('ignore')

In [1]:
import os

for var in ["HTTP_PROXY", "HTTPS_PROXY", "http_proxy", "https_proxy"]:
    if var in os.environ:
        del os.environ[var]

os.environ["NO_PROXY"] = "*"

In [2]:
from langchain_community.llms import Ollama
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import TextLoader,PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain.chains import RetrievalQA

In [3]:
filepath = "./data/vector-db/Attention all you need.pdf"

In [4]:
loader = PyPDFLoader(filepath)

In [5]:
docs = loader.load()

In [6]:
splitter = RecursiveCharacterTextSplitter(
                        chunk_size=150,
                        chunk_overlap=90
                )

In [7]:
chunks = splitter.split_documents(docs)

In [17]:
# chunks

In [10]:
embedding = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

In [11]:
vector_db = Chroma.from_documents(
                    documents = chunks,
                    embedding = embedding,
                    collection_name = "rag-pdf"
                )

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [12]:
retriever = vector_db.as_retriever(search_kwargs={'k': 3})

In [20]:
llm = Ollama(
    model="llama3.1",
    temperature=2
)

In [21]:
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    verbose=True
)

In [22]:
query = "What is multihead attention"

In [23]:
response = qa.run(query)



> Entering new RetrievalQA chain...

> Finished chain.


In [19]:
response

'According to the provided context, Multi-Head Attention (MHA) is a technique used by the model to jointly attend to information from different representations. In other words, it\'s a way for the model to consider multiple "heads" or sources of information simultaneously, rather than averaging weighted positions as mentioned earlier.\n\nThe specific ways in which MHA is applied in the model are described in section 3.2.3 Applications of Attention in our Model, but that context does not add any additional detail about what multi-head attention specifically means beyond what\'s mentioned above.'

In [24]:
response

'Multi-Head Attention is a mechanism that allows the model to jointly attend to information from different representation subspaces at different positions, counteracting an effect described in section 3.2.'